# 🐝 Lab 2.1 — Build Your Own Autograd Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varoonsahgal/advancedml/blob/main/notebooks/lab04_autograd_from_scratch.ipynb)

**⏱ Time:** ~2 hours  **🐝 Guide:** Buzz the Backprop Bee

### 🎯 Learning objectives
By the end of this lab you will be able to:
- Explain **reverse-mode automatic differentiation** as "the chain rule, applied on a graph, in reverse topological order".
- Implement a scalar `Value` autograd engine (micrograd-style) with a **topological sort** and a `backward()` pass.
- **Visualize** a computation graph and read gradients off it like a pro.
- **Gradient-check** any op against finite differences — the debugging superpower of every framework author.
- Extend the engine to **numpy tensors**, including the infamous *broadcasting-aware* backward (`unbroadcast`).
- Train an MLP on `make_moons` with *your* engine and prove its gradients match **PyTorch** to ~1e-12.

### 🧰 Prerequisites
Python classes & closures, numpy broadcasting, derivatives / chain rule, a first PyTorch training loop.

### 🌍 Why this matters in 2026
Every frontier model — from billion-parameter LLMs to diffusion video models — is trained by the same ~200-line idea you will write today. Frameworks (PyTorch, JAX, Triton kernels with custom backward passes) hide it, but the moment you write a custom CUDA/Triton op, a fused attention kernel, a new activation, a memory-saving checkpointing scheme, or debug a NaN in a 1000-GPU run, you are *reasoning about the backward graph*. Knowing autograd from the inside turns "the framework is magic" into "I know exactly which tensor is holding which gradient, and why". 🍯

## 🗺️ Roadmap

| Part | What you'll do | ⏱ min |
|---|---|---|
| 0 | Setup | 5 |
| 1 | Write a scalar `Value` class with `+`, `*` and a topological-sort `backward()` | 15 |
| 2 | Draw computation graphs; add `tanh`, `exp`, `relu`, `**` (🧪 #1) | 15 |
| 3 | Gradient-check against finite differences; meet the "`+=` bug" | 15 |
| 4 | Head-to-head with PyTorch's autograd | 10 |
| 5 | Upgrade to numpy tensors: `unbroadcast` (🧪 #2) and matmul backward (🧪 #3) | 30 |
| 6 | Train MLPs on two-moons with both engines (🧪 #4), and watch the boundary evolve 🎬 | 20 |
| 7 | Prove your gradients equal PyTorch's to ~1e-16 | 10 |

## 0. Setup
Everything here is preinstalled on Colab. `graphviz` is optional — if it is missing we fall back to a pure-matplotlib graph drawer.

In [ ]:
import os, math, random, time
import numpy as np
import torch
import matplotlib.pyplot as plt

FAST = os.environ.get("LAB_FAST", "0") == "1"   # smoke-test mode (normal users: leave unset)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(0)

HIVE = ["#F5B700", "#1F1B16", "#FF6B5B", "#1FA39A", "#3D4DB7", "#9B5DE5"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=HIVE), "figure.facecolor": "#FFFBEA",
                     "axes.facecolor": "#FFFDF5", "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 11})
print(f"torch {torch.__version__} | device={DEVICE} | FAST={FAST}")

import shutil
from sklearn.datasets import make_moons
torch.set_default_dtype(torch.float64)   # we compare to PyTorch at ~1e-12, so use float64 everywhere

try:
    import graphviz
    HAS_GRAPHVIZ = shutil.which("dot") is not None
except ImportError:
    HAS_GRAPHVIZ = False
print("graphviz available:", HAS_GRAPHVIZ)

## Part 1 — The chain rule, on a graph 🕸️

### Intuition
Any computation — a loss of a neural network included — is a **directed acyclic graph (DAG)** of tiny primitive operations (`+`, `*`, `tanh`, ...). Each primitive knows only one thing: its **local derivative**. Backprop just multiplies local derivatives along every path from the output back to each input, and **sums over paths**.

### The math
Let $L$ be the final scalar output and $v$ some intermediate node. Define the *adjoint* (a.k.a. "the grad") $\bar v = \partial L / \partial v$. If $v$ feeds into children $u_1, \dots, u_k$, the multivariate chain rule says

$$\bar v \;=\; \sum_{i=1}^{k} \bar u_i \,\frac{\partial u_i}{\partial v}.$$

Two consequences drive the whole design:
1. **We must finish computing $\bar u_i$ for every child before using it** → process nodes in *reverse topological order*.
2. **Gradients accumulate** (`+=`) because a node can feed many children — the sum above.

Why *reverse* mode? For $f:\mathbb{R}^n\to\mathbb{R}$ (millions of parameters → one loss), one reverse sweep gives all $n$ partials for roughly the cost of ~2–3 forward passes. Forward mode would need $n$ sweeps. That asymmetry is why deep learning is possible at all.

| op | forward | local derivatives |
|---|---|---|
| $c = a + b$ | $a+b$ | $\partial c/\partial a = 1,\ \partial c/\partial b = 1$ |
| $c = a \cdot b$ | $ab$ | $\partial c/\partial a = b,\ \partial c/\partial b = a$ |
| $c = a^n$ | $a^n$ | $n a^{n-1}$ |
| $c = \tanh a$ | $\tanh a$ | $1 - c^2$ |
| $c = e^a$ | $e^a$ | $c$ |
| $c = \mathrm{relu}(a)$ | $\max(0,a)$ | $\mathbb{1}[a>0]$ |

In [ ]:
class Value:
    """A scalar that remembers how it was made, so it can backprop through its history."""

    def __init__(self, data, _children=(), _op="", label=""):
        self.data = float(data)
        self.grad = 0.0                 # dL/d(self), filled in by backward()
        self._backward = lambda: None   # closure that pushes self.grad to the children
        self._prev = tuple(_children)   # the nodes this one was computed from
        self._op = _op                  # for visualization
        self.label = label

    def __repr__(self):
        return f"Value({self.label + '=' if self.label else ''}{self.data:.4f}, grad={self.grad:.4f})"

    # --- primitive ops -----------------------------------------------------
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += 1.0 * out.grad      # NOTE the +=  (gradients accumulate!)
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    # --- sugar built from primitives (no new backward code needed!) -------
    def __neg__(self):            return self * -1
    def __sub__(self, other):     return self + (-other)
    def __radd__(self, other):    return self + other
    def __rmul__(self, other):    return self * other
    def __rsub__(self, other):    return (-self) + other
    def __truediv__(self, other): return self * other ** -1     # needs __pow__ (Your Turn #1)
    def __rtruediv__(self, other):return other * self ** -1

    # --- the engine --------------------------------------------------------
    def topo_order(self):
        """Iterative post-order DFS: every node appears AFTER all nodes it depends on.
        (Iterative, so deep graphs don't hit Python's recursion limit.)"""
        topo, visited, stack = [], set(), [(self, False)]
        while stack:
            node, children_done = stack.pop()
            if children_done:
                topo.append(node)
                continue
            if node in visited:
                continue
            visited.add(node)
            stack.append((node, True))
            for child in node._prev:
                if child not in visited:
                    stack.append((child, False))
        return topo

    def backward(self):
        topo = self.topo_order()
        for v in topo:
            v.grad = 0.0           # fresh gradients for this pass
        self.grad = 1.0            # dL/dL = 1
        for v in reversed(topo):   # outputs first, leaves last
            v._backward()

# quick smoke test:  L = (a*b + c) * f
a, b, c, f = Value(2.0, label="a"), Value(-3.0, label="b"), Value(10.0, label="c"), Value(-2.0, label="f")
e = a * b; e.label = "e"
d = e + c; d.label = "d"
L = d * f; L.label = "L"
L.backward()
for v in (a, b, c, f, e, d, L):
    print(v)

Check by hand (Buzz insists 🐝): $L = (ab + c)f$, so $\partial L/\partial a = bf = (-3)(-2) = 6$, $\partial L/\partial b = af = -4$, $\partial L/\partial c = f = -2$, $\partial L/\partial f = ab + c = 4$. ✅

> **Recursive vs iterative topo sort.** The textbook version is a 6-line recursive DFS (`build(v)`: visit children, then append `v`). It's lovely, but a 10k-node graph blows Python's default recursion limit (1000). We use an explicit stack: push `(node, False)` to *expand* it, `(node, True)` to *emit* it after its children.

> **🍯 Key takeaway:** Backprop is just the chain rule applied node-by-node in **reverse topological order**, with gradients **accumulated** (`+=`) over every path from the loss to a node.

## Part 2 — Seeing the graph 👀

A computation graph is much easier to reason about when you can *see* it. Below, `draw_graph` uses graphviz when available (Colab has it) and otherwise a small matplotlib layered layout. Rectangles are values (`label | data | grad`), circles are the ops that produced them.

In [ ]:
def _graph_nodes_edges(root):
    nodes = root.topo_order()
    edges = [(child, n) for n in nodes for child in n._prev]
    return nodes, edges

def draw_graph_graphviz(root):
    dot = graphviz.Digraph(format="svg", graph_attr={"rankdir": "LR"})
    nodes, edges = _graph_nodes_edges(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(uid, label=f"{{ {n.label or ' '} | data {n.data:.3f} | grad {n.grad:.3f} }}",
                 shape="record", style="filled", fillcolor="#FFF3C4")
        if n._op:
            dot.node(uid + n._op, label=n._op, shape="circle", style="filled", fillcolor="#F5B700")
            dot.edge(uid + n._op, uid)
    for child, parent in edges:
        dot.edge(str(id(child)), str(id(parent)) + parent._op)
    return dot

def draw_graph_matplotlib(root, figsize=None):
    nodes, edges = _graph_nodes_edges(root)
    depth = {}
    for n in nodes:                          # topo order => children already have a depth
        depth[n] = 0 if not n._prev else 1 + max(depth[c] for c in n._prev)
    cols = {}
    for n in nodes:
        cols.setdefault(depth[n], []).append(n)
    pos = {n: (2.2 * d, -1.3 * i + 0.65 * (len(col) - 1)) for d, col in cols.items() for i, n in enumerate(col)}
    maxd = max(cols); maxh = max(len(c) for c in cols.values())
    fig, ax = plt.subplots(figsize=figsize or (2.6 * (maxd + 1) + 1, 1.0 * maxh + 1.5))
    for child, parent in edges:
        (x0, y0), (x1, y1) = pos[child], pos[parent]
        ax.annotate("", xy=(x1 - 0.75, y1), xytext=(x0 + 0.75, y0),
                    arrowprops=dict(arrowstyle="->", color="#1F1B16", lw=1.2))
    for n, (x, y) in pos.items():
        ax.text(x, y, f"{n.label or '·'}\ndata {n.data:.3f}\ngrad {n.grad:.3f}", ha="center", va="center",
                fontsize=9, bbox=dict(boxstyle="round,pad=0.35", fc="#FFF3C4", ec="#F5B700", lw=1.5))
        if n._op:
            ax.text(x - 0.95, y + 0.42, n._op, ha="center", va="center", fontsize=10, fontweight="bold",
                    bbox=dict(boxstyle="circle,pad=0.2", fc="#F5B700", ec="#1F1B16"))
    ax.set_xlim(-1.2, 2.2 * maxd + 1.2); ax.set_ylim(-0.65 * (maxh - 1) - 0.9, 0.65 * (maxh - 1) + 0.9)
    ax.axis("off"); ax.set_title(f"computation graph of {root.label or 'output'} (op badge = how the node was made)")
    plt.show()

def draw_graph(root):
    if HAS_GRAPHVIZ:
        from IPython.display import display
        display(draw_graph_graphviz(root))
    else:
        draw_graph_matplotlib(root)

draw_graph(L)

Read the graph right-to-left like the backward pass does: $L$ has grad 1; the `*` node hands $f$ the value of $d$ (=4) and hands $d$ the value of $f$ (= −2); the `+` node simply *routes* $d$'s gradient unchanged to $e$ and $c$ (a `+` is a **gradient distributor**); the `*` node *swaps* values (a `*` is a **gradient swapper/scaler**). These little mental rules ("add distributes, mul swaps, max routes") make you fast at debugging backward passes.

### 🧪 Your Turn #1 — teach `Value` some nonlinearities
Implement the forward **and** backward of `__pow__` (constant exponent), `exp`, `tanh`, and `relu`.

**Hints**
- Follow the pattern of `__mul__`: compute `out`, define a closure `_backward` that does `self.grad += <local derivative> * out.grad`, attach it, return `out`.
- `tanh` and `exp` have derivatives that can be written in terms of `out.data` — reuse it, it's cheaper.
- For `relu`, the derivative at exactly 0 is a convention; use `out.data > 0`.

In [ ]:
# 🧪 YOUR TURN: fill in the TODOs, then run the checkpoint further below.
def _pow(self, n):
    assert isinstance(n, (int, float)), "only constant exponents"
    out = Value(self.data ** n, (self,), f"**{n}")
    def _backward():
        self.grad += 0.0   # TODO: n * x^(n-1) * out.grad
    out._backward = _backward
    return out

def _exp(self):
    out = Value(math.exp(self.data), (self,), "exp")
    def _backward():
        pass               # TODO
    out._backward = _backward
    return out

def _tanh(self):
    t = math.tanh(self.data)
    out = Value(t, (self,), "tanh")
    def _backward():
        pass               # TODO
    out._backward = _backward
    return out

def _relu(self):
    out = Value(max(0.0, self.data), (self,), "relu")
    def _backward():
        pass               # TODO
    out._backward = _backward
    return out

Value.__pow__, Value.exp, Value.tanh, Value.relu = _pow, _exp, _tanh, _relu

In [ ]:
# ✅ Solution (peek only after trying!)
def _pow(self, n):
    assert isinstance(n, (int, float)), "only constant exponents"
    out = Value(self.data ** n, (self,), f"**{n}")
    def _backward():
        self.grad += n * self.data ** (n - 1) * out.grad
    out._backward = _backward
    return out

def _exp(self):
    out = Value(math.exp(self.data), (self,), "exp")
    def _backward():
        self.grad += out.data * out.grad          # d/dx e^x = e^x
    out._backward = _backward
    return out

def _tanh(self):
    t = math.tanh(self.data)
    out = Value(t, (self,), "tanh")
    def _backward():
        self.grad += (1 - t ** 2) * out.grad      # d/dx tanh = 1 - tanh^2
    out._backward = _backward
    return out

def _relu(self):
    out = Value(max(0.0, self.data), (self,), "relu")
    def _backward():
        self.grad += (out.data > 0) * out.grad
    out._backward = _backward
    return out

Value.__pow__, Value.exp, Value.tanh, Value.relu = _pow, _exp, _tanh, _relu

Now a real neuron: $o = \tanh(x_1 w_1 + x_2 w_2 + b)$. (The bias $b = 6.8813735870195432$ is chosen so the pre-activation is ≈ 0.8814, i.e. $o \approx 0.7071 = 1/\sqrt2$.)

In [ ]:
x1, x2 = Value(2.0, label="x1"), Value(0.0, label="x2")
w1, w2 = Value(-3.0, label="w1"), Value(1.0, label="w2")
b = Value(6.8813735870195432, label="b")
x1w1 = x1 * w1; x1w1.label = "x1w1"
x2w2 = x2 * w2; x2w2.label = "x2w2"
s = x1w1 + x2w2; s.label = "x1w1+x2w2"
n = s + b; n.label = "n"
o = n.tanh(); o.label = "o"
o.backward()
draw_graph(o)
print("dO/dw1 =", w1.grad, "(expected x1*(1-o^2) =", 2.0 * (1 - o.data**2), ")")

> **🍯 Key takeaway:** Every op only needs to know its *local* derivative — `+` distributes the upstream gradient, `*` swaps and scales it, nonlinearities multiply by their slope. The graph does the rest.

## Part 3 — Trust, but verify: gradient checking 🔍

### Intuition
Analytic gradients are fast but easy to get wrong (a forgotten `+=`, a transposed matrix...). Numerical gradients are slow but almost impossible to get wrong. So we **check one against the other**.

### The math
The **central difference** has error $O(h^2)$ (vs $O(h)$ for the one-sided version):

$$\frac{\partial f}{\partial x_i} \approx \frac{f(x + h e_i) - f(x - h e_i)}{2h}.$$

Compare with a **relative error** so the test is scale-free:

$$\text{rel\_err} = \frac{|g_{\text{analytic}} - g_{\text{numeric}}|}{\max(|g_{\text{analytic}}| + |g_{\text{numeric}}|,\ \varepsilon)}.$$

Rule of thumb in float64 with $h \approx 10^{-6}$: rel_err ≲ 1e-7 is great, ~1e-4 is suspicious, > 1e-2 is a bug (unless you hit a kink like `relu` at 0).

In [ ]:
def grad_check(fn, xs, h=1e-6, verbose=True):
    """fn: python function taking a list of Values and returning a Value. xs: list of floats.
    Returns max relative error between autograd and central differences."""
    vals = [Value(x) for x in xs]
    out = fn(vals)
    out.backward()
    analytic = [v.grad for v in vals]
    numeric = []
    for i in range(len(xs)):
        xp = list(xs); xp[i] += h
        xm = list(xs); xm[i] -= h
        numeric.append((fn([Value(x) for x in xp]).data - fn([Value(x) for x in xm]).data) / (2 * h))
    errs = [abs(a - n) / max(abs(a) + abs(n), 1e-12) for a, n in zip(analytic, numeric)]
    if verbose:
        for i, (a, n, e) in enumerate(zip(analytic, numeric, errs)):
            print(f"  x{i}: autograd={a: .8f}  numeric={n: .8f}  rel_err={e:.2e}")
    return max(errs)

def messy(v):
    x, y, z = v
    return ((x * y).tanh() + (z ** 3) / (x + 2.5) - (y * z).exp() * 0.1 + (x - z).relu() * y) ** 2

err = grad_check(messy, [0.3, -1.2, 0.7])
print("max relative error:", err)

In [ ]:
# ✔️ Checkpoint — your nonlinearities pass the gradient check
rng = np.random.default_rng(0)
for trial in range(20):
    xs = list(rng.uniform(-1.5, 1.5, size=3))
    assert grad_check(messy, xs, verbose=False) < 1e-6, f"gradient mismatch at {xs}"
for op in ["tanh", "exp", "relu"]:
    assert grad_check(lambda v: getattr(v[0] * 1.7, op)(), [0.37], verbose=False) < 1e-6, op
assert grad_check(lambda v: v[0] ** 3 + v[0] ** -2, [1.3], verbose=False) < 1e-6
print("✔️ All gradient checks passed — Buzz is proud of you! 🐝")

### 🔮 Predict-then-run: the "+= bug"
Consider `b = a + a` with `a = 3`. Clearly $db/da = 2$.

**Predict:** if the `__add__` backward used `self.grad = out.grad` (assignment) instead of `+=`, what would `a.grad` be? What about `c = a * a`? Write your guesses down, then run the cell.

In [ ]:
class BuggyValue(Value):
    def __add__(self, other):
        other = other if isinstance(other, Value) else BuggyValue(other)
        out = BuggyValue(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad = out.grad      # 🐛 assignment instead of accumulation
            other.grad = out.grad
        out._backward = _backward
        return out
    def __mul__(self, other):
        other = other if isinstance(other, Value) else BuggyValue(other)
        out = BuggyValue(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad = other.data * out.grad
            other.grad = self.data * out.grad
        out._backward = _backward
        return out

for Cls in (Value, BuggyValue):
    a = Cls(3.0); (a + a).backward(); g_add = a.grad
    a = Cls(3.0); (a * a).backward(); g_mul = a.grad
    print(f"{Cls.__name__:>10}:  d(a+a)/da = {g_add}   d(a*a)/da = {g_mul}   (truth: 2 and 6)")

When the same node is used twice, each use is a separate path in the graph and the chain rule says **sum over paths**. Assignment overwrites the first path's contribution. In PyTorch this is exactly why `.grad` *accumulates* across `backward()` calls and you must call `optimizer.zero_grad()` — the accumulation is a feature (it enables gradient accumulation over micro-batches), not a bug. (Our toy `backward()` re-zeroes the grads of its graph at the start of each call for convenience; PyTorch deliberately does not — that's *your* job via `zero_grad()`.)

> **🍯 Key takeaway:** Central differences + relative error are the universal lie detector for gradients — run them on every custom op you ever write.

## Part 4 — Head-to-head with PyTorch 🥊

PyTorch's autograd is a production-grade version of exactly what we built: each tensor op records a `grad_fn` node, and `.backward()` walks them in reverse topological order. Let's run the same messy function through both.

In [ ]:
def messy_torch(x, y, z):
    return (torch.tanh(x * y) + z ** 3 / (x + 2.5) - torch.exp(y * z) * 0.1 + torch.relu(x - z) * y) ** 2

xs = [0.3, -1.2, 0.7]
tx = [torch.tensor(v, requires_grad=True) for v in xs]
out_t = messy_torch(*tx); out_t.backward()
vx = [Value(v) for v in xs]
out_v = messy(vx); out_v.backward()

print(f"forward : ours={out_v.data:.15f}  torch={out_t.item():.15f}")
for i in range(3):
    print(f"grad x{i}: ours={vx[i].grad:.15f}  torch={tx[i].grad.item():.15f}")
print("PyTorch's graph (grad_fn chain head):", out_t.grad_fn, "->", out_t.grad_fn.next_functions[0][0])

# ✔️ Checkpoint
assert abs(out_v.data - out_t.item()) < 1e-12
assert all(abs(vx[i].grad - tx[i].grad.item()) < 1e-12 for i in range(3))
print("✔️ Identical to PyTorch to 1e-12.")

> **🍯 Key takeaway:** PyTorch's autograd is the same idea as your `Value` class (a `grad_fn` graph walked in reverse) — just in C++, on tensors, with many more ops.

## Part 5 — From scalars to tensors (and the broadcasting trap) 🧊

A scalar engine creates one Python object per multiply — hopeless for real networks. Real frameworks make each node a **whole array** and each op a vectorized kernel. The backward rules become matrix calculus:

| op | forward | backward (given upstream grad $G = \partial L/\partial \text{out}$) |
|---|---|---|
| `C = A + B` | elementwise | $\bar A \mathrel{+}= G,\ \bar B \mathrel{+}= G$ *(then un-broadcast!)* |
| `C = A * B` | elementwise | $\bar A \mathrel{+}= G \odot B,\ \bar B \mathrel{+}= G \odot A$ *(then un-broadcast!)* |
| `C = A @ B` | $(n{\times}k)(k{\times}m)$ | $\bar A \mathrel{+}= G B^\top,\ \bar B \mathrel{+}= A^\top G$ |
| `s = A.sum(axis)` | reduce | $\bar A \mathrel{+}= \text{broadcast}(G)$ |

### The broadcasting trap
If `b` has shape `(5,)` and `X @ W` has shape `(32, 5)`, then `X @ W + b` silently **copies** `b` 32 times. The chain rule says the gradient of a copied variable is the **sum** of the gradients of the copies. So the gradient flowing back, of shape `(32, 5)`, must be **summed** back down to shape `(5,)`.

Numpy's broadcasting rules: (1) left-pad the smaller shape with 1s; (2) any dimension of size 1 is stretched. So `unbroadcast(grad, shape)` must (1) sum away the extra leading axes and (2) sum (with `keepdims=True`) over axes where the original had size 1.

### 🧪 Your Turn #2 — `unbroadcast`
Implement `unbroadcast(grad, shape)` that reduces `grad` (shape after broadcasting) to `shape` (shape before).

**Hints:** a `while grad.ndim > len(shape)` loop that sums axis 0; then `for i, s in enumerate(shape): if s == 1 and grad.shape[i] != 1: ...sum(axis=i, keepdims=True)`.

In [ ]:
# 🧪 YOUR TURN
def unbroadcast(grad, shape):
    # TODO 1: sum out the leading axes that broadcasting added
    # TODO 2: sum (keepdims=True) over axes where the original shape had size 1
    return grad

In [ ]:
# ✅ Solution (peek only after trying!)
def unbroadcast(grad, shape):
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for i, s in enumerate(shape):
        if s == 1 and grad.shape[i] != 1:
            grad = grad.sum(axis=i, keepdims=True)
    return grad

In [ ]:
# ✔️ Checkpoint — fails on the unchanged TODO stub
g = np.ones((4, 3, 5))
assert unbroadcast(g, (5,)).shape == (5,) and np.all(unbroadcast(g, (5,)) == 12)
assert unbroadcast(g, (3, 1)).shape == (3, 1) and np.all(unbroadcast(g, (3, 1)) == 20)
assert unbroadcast(g, (1, 3, 5)).shape == (1, 3, 5)
assert unbroadcast(np.ones((2, 2)), ()).shape == () and unbroadcast(np.ones((2, 2)), ()) == 4
print("✔️ unbroadcast works")

### 🧪 Your Turn #3 — matmul backward
For $C = AB$ with $A \in \mathbb{R}^{n\times k}$, $B \in \mathbb{R}^{k\times m}$ and upstream $G = \partial L/\partial C \in \mathbb{R}^{n \times m}$, derive $\partial L/\partial A$ and $\partial L/\partial B$.

**Hint:** $C_{ij} = \sum_r A_{ir} B_{rj}$, so $\partial L/\partial A_{ir} = \sum_j G_{ij} B_{rj}$. Then **check shapes**: the result must be $n\times k$ — there's only one way to multiply $G$ ($n\times m$) and $B$ ($k\times m$) to get that.

In [ ]:
# 🧪 YOUR TURN
def matmul_backward(A, B, G):
    dA = np.zeros_like(A)   # TODO
    dB = np.zeros_like(B)   # TODO
    return dA, dB

In [ ]:
# ✅ Solution (peek only after trying!)
def matmul_backward(A, B, G):
    dA = G @ B.T      # (n,m)(m,k) -> (n,k)
    dB = A.T @ G      # (k,n)(n,m) -> (k,m)
    return dA, dB

Now the tensor engine itself. Same skeleton as `Value` — `data`, `grad`, `_prev`, `_backward`, topological `backward()` — but everything is a numpy array, and every elementwise backward goes through `unbroadcast`. We also add a fused, numerically stable `log_softmax` (backward: $\bar x = G - \mathrm{softmax}(x)\,\textstyle\sum G$).

In [ ]:
class Tensor:
    def __init__(self, data, _children=(), _op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.grad = np.zeros_like(self.data)
        self._backward = lambda: None
        self._prev = tuple(_children)
        self._op = _op

    shape = property(lambda self: self.data.shape)
    def __repr__(self): return f"Tensor(shape={self.shape}, op='{self._op}')"

    @staticmethod
    def _lift(x): return x if isinstance(x, Tensor) else Tensor(x)

    # ----- elementwise (broadcasting-aware) -----
    def __add__(self, other):
        other = Tensor._lift(other)
        out = Tensor(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += unbroadcast(out.grad, self.shape)
            other.grad += unbroadcast(out.grad, other.shape)
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = Tensor._lift(other)
        out = Tensor(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += unbroadcast(other.data * out.grad, self.shape)
            other.grad += unbroadcast(self.data * out.grad, other.shape)
        out._backward = _backward
        return out

    def __pow__(self, n):
        out = Tensor(self.data ** n, (self,), f"**{n}")
        def _backward():
            self.grad += n * self.data ** (n - 1) * out.grad
        out._backward = _backward
        return out

    __radd__ = __add__
    __rmul__ = __mul__
    def __neg__(self): return self * -1.0
    def __sub__(self, other): return self + (-Tensor._lift(other))
    def __rsub__(self, other): return Tensor._lift(other) + (-self)
    def __truediv__(self, other): return self * Tensor._lift(other) ** -1

    # ----- linear algebra -----
    def __matmul__(self, other):
        out = Tensor(self.data @ other.data, (self, other), "@")
        def _backward():
            dA, dB = matmul_backward(self.data, other.data, out.grad)
            self.grad += dA
            other.grad += dB
        out._backward = _backward
        return out

    # ----- reductions -----
    def sum(self, axis=None, keepdims=False):
        out = Tensor(self.data.sum(axis=axis, keepdims=keepdims), (self,), "sum")
        def _backward():
            g = out.grad
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            self.grad += np.broadcast_to(g, self.shape)
        out._backward = _backward
        return out

    def mean(self, axis=None):
        n = self.data.size if axis is None else self.data.shape[axis]
        return self.sum(axis=axis) * (1.0 / n)

    # ----- nonlinearities -----
    def relu(self):
        out = Tensor(np.maximum(self.data, 0), (self,), "relu")
        def _backward(): self.grad += (self.data > 0) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = np.tanh(self.data)
        out = Tensor(t, (self,), "tanh")
        def _backward(): self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Tensor(np.exp(self.data), (self,), "exp")
        def _backward(): self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def log(self):
        out = Tensor(np.log(self.data), (self,), "log")
        def _backward(): self.grad += out.grad / self.data
        out._backward = _backward
        return out

    def log_softmax(self, axis=-1):
        m = self.data.max(axis=axis, keepdims=True)
        lse = m + np.log(np.exp(self.data - m).sum(axis=axis, keepdims=True))
        out = Tensor(self.data - lse, (self,), "log_softmax")
        def _backward():
            sm = np.exp(out.data)
            self.grad += out.grad - sm * out.grad.sum(axis=axis, keepdims=True)
        out._backward = _backward
        return out

    def gather_rows(self, idx):
        """out[i] = self[i, idx[i]] — used for picking the log-prob of the correct class."""
        idx = np.asarray(idx)
        rows = np.arange(len(idx))
        out = Tensor(self.data[rows, idx], (self,), "gather")
        def _backward():
            g = np.zeros_like(self.data)
            np.add.at(g, (rows, idx), out.grad)
            self.grad += g
        out._backward = _backward
        return out

    # ----- engine (identical idea to Value) -----
    def backward(self):
        topo, visited, stack = [], set(), [(self, False)]
        while stack:
            node, done = stack.pop()
            if done: topo.append(node); continue
            if node in visited: continue
            visited.add(node); stack.append((node, True))
            stack.extend((c, False) for c in node._prev if c not in visited)
        for v in topo:
            v.grad = np.zeros_like(v.data)
        assert self.data.size == 1, "backward() starts from a scalar loss"
        self.grad = np.ones_like(self.data)
        for v in reversed(topo):
            v._backward()

def cross_entropy(logits, y):
    return -(logits.log_softmax(axis=1).gather_rows(y)).mean()

### ✔️ Checkpoint — tensor engine vs PyTorch, with nasty broadcasting
We build an expression with shapes `(8,3) @ (3,5) + (5,)`, then multiply by a `(1,5)` row, a `(8,1)` column and a scalar, push it through `tanh`, `exp`, `log`, `relu`, a sum over an axis, and cross-entropy. Every gradient must match PyTorch.

In [ ]:
rng = np.random.default_rng(1)
arrs = dict(X=rng.normal(size=(8, 3)), W=rng.normal(size=(3, 5)), b=rng.normal(size=(5,)),
            row=rng.normal(size=(1, 5)), col=rng.normal(size=(8, 1)), s=np.array(0.7))
y = rng.integers(0, 5, size=8)

def expr(T, ops):
    """Same expression written once, run with either engine. `ops` abstracts the few API differences."""
    h = ops["tanh"]((T["X"] @ T["W"] + T["b"]) * T["row"])
    h = h * T["col"] + T["s"] * ops["relu"](h)
    z = ops["log"](ops["exp"](h) + 1.0)                   # softplus written the long way
    logits = z - ops["sum1"](z) / 5.0
    return ops["ce"](logits, y) + ops["sumall"](T["W"] ** 2) * 0.01

ours = {k: Tensor(v) for k, v in arrs.items()}
loss_ours = expr(ours, dict(tanh=Tensor.tanh, relu=Tensor.relu, log=Tensor.log, exp=Tensor.exp,
                            sum1=lambda t: t.sum(axis=1, keepdims=True), sumall=Tensor.sum, ce=cross_entropy))
loss_ours.backward()

th = {k: torch.tensor(v, requires_grad=True) for k, v in arrs.items()}
loss_th = expr(th, dict(tanh=torch.tanh, relu=torch.relu, log=torch.log, exp=torch.exp,
                        sum1=lambda t: t.sum(dim=1, keepdim=True), sumall=torch.sum,
                        ce=lambda lg, yy: torch.nn.functional.cross_entropy(lg, torch.tensor(yy))))
loss_th.backward()

print(f"loss: ours={loss_ours.data.item():.12f}  torch={loss_th.item():.12f}")
for k in arrs:
    diff = np.abs(ours[k].grad - th[k].grad.numpy()).max()
    print(f"  grad[{k:>3}] shape={str(ours[k].grad.shape):8s} max|diff|={diff:.2e}")
    assert ours[k].grad.shape == arrs[k].shape, f"{k}: gradient has wrong shape (unbroadcast bug?)"
    assert diff < 1e-10, k
assert abs(loss_ours.data.item() - loss_th.item()) < 1e-12
print("✔️ Tensor engine matches PyTorch on every gradient (including broadcast ones)!")

> **🍯 Key takeaway:** Broadcasting copies values in the forward pass, so the backward pass must **sum** gradients back to the original shape; and for matmul, $\bar A = GB^\top$, $\bar B = A^\top G$ — check the shapes and you can't go wrong.

## Part 6 — Train an MLP on `make_moons` with your engine 🌙

Time to fly. We'll train two models on the classic two-moons dataset:
1. **Scalar engine** (`Value`): a tiny 2→16→16→1 MLP with a max-margin (hinge) loss — every weight is a Python object, so it's slow but transparent.
2. **Tensor engine** (`Tensor`): a 2→32→32→2 MLP with softmax cross-entropy — vectorized and quick.

Hinge loss for labels $y\in\{-1,+1\}$ and score $s$: $\ \ell = \max(0, 1 - y s)$, plus L2 regularization $\lambda \lVert\theta\rVert^2$.

In [ ]:
X_moons, y_moons = make_moons(n_samples=100 if not FAST else 50, noise=0.1, random_state=0)
y_pm = y_moons * 2 - 1     # {-1, +1} for the hinge loss

class Neuron:
    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]   # micrograd-style init
        self.b = Value(0.0)
        self.nonlin = nonlin
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.relu() if self.nonlin else act
    def parameters(self): return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout, **kw): self.neurons = [Neuron(nin, **kw) for _ in range(nout)]
    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out
    def parameters(self): return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1], nonlin=(i != len(nouts) - 1)) for i in range(len(nouts))]
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    def parameters(self): return [p for l in self.layers for p in l.parameters()]

random.seed(0)
scalar_model = MLP(2, [16, 16, 1])
print("scalar MLP parameters:", len(scalar_model.parameters()))

def hinge_loss(model, X, y, alpha=1e-4):
    scores = [model(list(map(Value, xrow))) for xrow in X]
    data_loss = sum(((1 + -float(yi) * si).relu() for yi, si in zip(y, scores)), Value(0.0)) * (1.0 / len(X))
    reg_loss = alpha * sum((p * p for p in model.parameters()), Value(0.0))
    acc = np.mean([(yi > 0) == (si.data > 0) for yi, si in zip(y, scores)])
    return data_loss + reg_loss, acc

### 🧪 Your Turn #4 — write the training step
Fill in the three classic lines of a training step: **backward**, **SGD update**, and (implicitly, since our `backward()` re-zeroes grads) nothing else! Use a learning rate that decays linearly from 1.0 to 0.1.

**Hints:** `loss.backward()`; then `for p in model.parameters(): p.data -= lr * p.grad`.

In [ ]:
# 🧪 YOUR TURN
def train_step(model, X, y, step, total_steps):
    loss, acc = hinge_loss(model, X, y)
    lr = 1.0 - 0.9 * step / total_steps
    # TODO: backprop
    # TODO: SGD update of every parameter
    return loss.data, acc

In [ ]:
# ✅ Solution (peek only after trying!)
def train_step(model, X, y, step, total_steps):
    loss, acc = hinge_loss(model, X, y)
    lr = 1.0 - 0.9 * step / total_steps
    loss.backward()
    for p in model.parameters():
        p.data -= lr * p.grad
    return loss.data, acc

### 🔮 Predict-then-run
The scalar engine builds a brand-new graph of tens of thousands of `Value` nodes **every step**. How long do you think one step takes? 1 ms? 100 ms? 1 s? And how many steps until 100% training accuracy?

In [ ]:
STEPS = 12 if FAST else 60
t0 = time.time()
for k in range(STEPS):
    loss, acc = train_step(scalar_model, X_moons, y_pm, k, STEPS)
    if k % 10 == 0 or k == STEPS - 1:
        print(f"step {k:3d}  loss {loss:.4f}  acc {acc:.2%}")
dt = (time.time() - t0) / STEPS
n_nodes = len(hinge_loss(scalar_model, X_moons, y_pm)[0].topo_order())
print(f"~{dt*1000:.0f} ms/step, graph has {n_nodes:,} Value nodes")

# ✔️ Checkpoint — a do-nothing train_step leaves accuracy near 50%
assert acc > 0.8, "accuracy barely moved: did train_step call backward() and update every parameter?"
print(f"✔️ Your training step works: {acc:.0%} train accuracy")

In [ ]:
def plot_boundary(predict_fn, X, y, title, ax):
    xx, yy = np.meshgrid(np.linspace(-1.6, 2.6, 120), np.linspace(-1.1, 1.6, 120))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = predict_fn(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[-1e9, 0, 1e9], colors=["#FFE08A", "#A6E3DD"], alpha=0.8)
    ax.contour(xx, yy, Z, levels=[0], colors="#1F1B16", linewidths=1.5)
    ax.scatter(X[:, 0], X[:, 1], c=np.where(y > 0, HIVE[3], HIVE[0]), edgecolor="#1F1B16", s=22)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

def scalar_predict(grid):
    # Evaluating 14,400 grid points through Value objects would build ~10M nodes, so we
    # copy the learned weights into numpy and run the *same* forward pass vectorized.
    h = grid
    for i, layer in enumerate(scalar_model.layers):
        W = np.array([[w.data for w in n.w] for n in layer.neurons])
        b_ = np.array([n.b.data for n in layer.neurons])
        h = h @ W.T + b_
        if i < len(scalar_model.layers) - 1:
            h = np.maximum(h, 0)
    return h[:, 0]

fig, ax = plt.subplots(figsize=(6, 4.5))
plot_boundary(scalar_predict, X_moons, y_pm, "Scalar engine (Value) MLP — trained by YOUR backprop", ax)
plt.show()

Now the vectorized engine. Same math, but each layer is one `@` and one broadcast `+ b` — this is where `unbroadcast` earns its keep (the bias gradient is a sum over the batch).

In [ ]:
class TensorMLP:
    def __init__(self, sizes, seed=0):
        rng = np.random.default_rng(seed)
        self.params = []
        for nin, nout in zip(sizes[:-1], sizes[1:]):
            W = Tensor(rng.normal(size=(nin, nout)) * math.sqrt(2.0 / nin))   # Kaiming/He init for ReLU
            b = Tensor(np.zeros(nout))
            self.params += [W, b]
    def __call__(self, X):
        h = X if isinstance(X, Tensor) else Tensor(X)
        n_layers = len(self.params) // 2
        for i in range(n_layers):
            W, b = self.params[2 * i], self.params[2 * i + 1]
            h = h @ W + b
            if i < n_layers - 1:
                h = h.relu()
        return h

Xm, ym = make_moons(n_samples=1000 if not FAST else 300, noise=0.12, random_state=1)
tmodel = TensorMLP([2, 32, 32, 2])
STEPS = 300 if FAST else 1500
hist, frames = [], []
gx, gy = np.meshgrid(np.linspace(-1.6, 2.6, 90), np.linspace(-1.1, 1.6, 90))
GRID = np.c_[gx.ravel(), gy.ravel()]
snap_steps = set(np.unique(np.geomspace(1, STEPS, 40).astype(int) - 1))   # log-spaced: early steps change most
for k in range(STEPS):
    loss = cross_entropy(tmodel(Xm), ym)
    loss.backward()
    for p in tmodel.params:
        p.data -= 0.3 * p.grad
    if k % 50 == 0 or k == STEPS - 1:
        acc = (tmodel(Xm).data.argmax(1) == ym).mean()
        hist.append((k, loss.data.item(), acc))
    if k in snap_steps:
        lg = tmodel(GRID).data
        frames.append((k, (lg[:, 1] - lg[:, 0]).reshape(gx.shape), loss.data.item()))
print(f"final loss {hist[-1][1]:.4f}, train acc {hist[-1][2]:.2%}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
plot_boundary(scalar_predict, X_moons, y_pm, "Scalar engine (Value) MLP", axes[0])
plot_boundary(lambda g: (lambda lg: lg[:, 1] - lg[:, 0])(tmodel(g).data), Xm, ym * 2 - 1,
              "Tensor engine MLP (2-32-32-2)", axes[1])
ks, ls, accs = zip(*hist)
axes[2].plot(ks, ls, label="loss"); axes[2].plot(ks, accs, label="train acc")
axes[2].set_xlabel("step"); axes[2].legend(); axes[2].set_title("Tensor-engine training curve")
plt.tight_layout(); plt.show()

> **🍯 Key takeaway:** Same math, wildly different speed: the scalar engine builds tens of thousands of Python objects per step, the tensor engine a few dozen arrays. Vectorization is why frameworks operate on tensors.

### 🎬 Watch your gradients sculpt the boundary
Every frame below is a snapshot of the tensor-engine MLP during training (log-spaced in time, because the early steps change the most). The colours are the model's two classes; the black line is the decision boundary — all of its motion is driven by gradients **your** `backward()` computed.

In [ ]:
from matplotlib import animation
from IPython.display import HTML, display

fig, ax = plt.subplots(figsize=(6.4, 4.2))
def draw(i):
    ax.clear()
    k, Z, l = frames[i]
    ax.contourf(gx, gy, Z, levels=[-1e9, 0, 1e9], colors=["#FFE08A", "#A6E3DD"], alpha=0.85)
    ax.contour(gx, gy, Z, levels=[0], colors="#1F1B16", linewidths=1.5)
    ax.scatter(Xm[:, 0], Xm[:, 1], c=np.where(ym > 0, HIVE[3], HIVE[0]), edgecolor="#1F1B16", s=12, lw=0.4)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"step {k + 1:5d}   loss {l:.3f}")
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=120)
plt.close(fig)
display(HTML(anim.to_jshtml(default_mode="once")))

## Part 7 — The moment of truth: identical gradients to PyTorch 🏁

"It trains" is not proof of correctness — buggy gradients often still train, just worse. The real test: load the **same weights** into a PyTorch model and compare every parameter gradient. We'll do it for **both** engines.

In [ ]:
# --- tensor engine vs torch.nn ---
torch_mlp = torch.nn.Sequential(torch.nn.Linear(2, 32), torch.nn.ReLU(), torch.nn.Linear(32, 32),
                                torch.nn.ReLU(), torch.nn.Linear(32, 2))
linears = [m for m in torch_mlp if isinstance(m, torch.nn.Linear)]
with torch.no_grad():
    for i, lin in enumerate(linears):
        lin.weight.copy_(torch.tensor(tmodel.params[2 * i].data.T))   # torch stores W as (out, in)
        lin.bias.copy_(torch.tensor(tmodel.params[2 * i + 1].data))

loss_ours = cross_entropy(tmodel(Xm), ym); loss_ours.backward()
loss_torch = torch.nn.functional.cross_entropy(torch_mlp(torch.tensor(Xm)), torch.tensor(ym))
torch_mlp.zero_grad(); loss_torch.backward()

max_diff = 0.0
for i, lin in enumerate(linears):
    max_diff = max(max_diff, np.abs(tmodel.params[2 * i].grad - lin.weight.grad.numpy().T).max(),
                   np.abs(tmodel.params[2 * i + 1].grad - lin.bias.grad.numpy()).max())
print(f"tensor engine: loss diff {abs(loss_ours.data.item() - loss_torch.item()):.2e}, max grad diff {max_diff:.2e}")

# --- scalar engine vs torch (hinge loss + L2) ---
params = scalar_model.parameters()
loss_v, _ = hinge_loss(scalar_model, X_moons, y_pm); loss_v.backward()
Ws, bs = [], []
for layer in scalar_model.layers:
    Ws.append(torch.tensor([[w.data for w in n.w] for n in layer.neurons], requires_grad=True))
    bs.append(torch.tensor([n.b.data for n in layer.neurons], requires_grad=True))
h = torch.tensor(X_moons)
for i in range(3):
    h = h @ Ws[i].T + bs[i]
    if i < 2: h = torch.relu(h)
scores = h.squeeze(1)
loss_t = torch.relu(1 - torch.tensor(y_pm, dtype=torch.float64) * scores).mean() \
         + 1e-4 * sum((W ** 2).sum() + (b_ ** 2).sum() for W, b_ in zip(Ws, bs))
loss_t.backward()
ours_flat = np.array([p.grad for p in params])
torch_flat = np.concatenate([np.concatenate([W.grad.numpy(), b_.grad.numpy()[:, None]], 1).ravel()
                             for W, b_ in zip(Ws, bs)])
scalar_diff = np.abs(ours_flat - torch_flat).max()
print(f"scalar engine: loss diff {abs(loss_v.data - loss_t.item()):.2e}, max grad diff {scalar_diff:.2e}")

# ✔️ Checkpoint
assert max_diff < 1e-10 and scalar_diff < 1e-10
print("✔️ Both of your engines produce PyTorch-identical gradients. You built autograd. 🐝🍯")

> **🍯 Key takeaway:** "It trains" is not evidence of correct gradients — matching a trusted reference on identical weights is.

## 🏆 Boss Level
1. **Higher-order gradients.** Make `backward()` itself build a graph (i.e., write `_backward` using `Value` ops instead of floats, with a `create_graph=True` flag) so you can compute $\partial^2 f/\partial x^2$. Verify against `torch.autograd.grad(..., create_graph=True)`. *(This is how MAML, Hessian-vector products and some influence-function methods work.)*
2. **Forward mode with dual numbers.** Implement a `Dual(a, b)` class where $b$ carries $\partial/\partial x$, and show that forward mode needs $n$ passes for an $n$-input function while reverse mode needs one. Then compute a Jacobian–vector product and compare to `torch.func.jvp`.
3. **Memory diet.** Add a `checkpoint(fn)` wrapper to the tensor engine that does *not* store intermediates in the forward pass and recomputes them during backward. Measure peak number of live arrays for a 20-layer MLP with and without it (activation checkpointing is standard practice in LLM training).

## 🍯 Key Takeaways
- Reverse-mode autodiff = **local derivatives + chain rule + reverse topological order**; one backward pass yields the gradient w.r.t. *every* input, for roughly the cost of a couple of forward passes.
- Gradients **accumulate** (`+=`) because a value used in several places receives gradient along each path.
- **Gradient checking** with central differences and relative error is the universal correctness test for custom ops.
- Broadcasting in the forward pass means **summing** in the backward pass — `unbroadcast` is the most common bug site in hand-written backward code.
- Matmul backward: $\bar A = G B^\top$, $\bar B = A^\top G$ — derive once, then just check shapes.
- Tensors, not scalars: the same algorithm becomes fast when each node is an array op.
- Your engines match PyTorch to ~1e-16 in float64: there's no magic, just bookkeeping. 🐝

## 🎤 Explain it to a friend
Imagine a factory line where each worker does one tiny step (add, multiply, squash) and the final product has a defect score. Explain to a friend — without equations — how you could figure out how much *each* worker's settings contributed to the defect by walking the line **backwards once**, and why a worker who feeds two downstream stations has to *add up* the blame coming back from both.

## 💼 Interview corner
**1. Why does deep learning use reverse-mode rather than forward-mode autodiff?**
<details><summary>Answer</summary>

For $f:\mathbb R^n\to\mathbb R^m$, forward mode costs one pass per *input* direction, reverse mode one pass per *output*. Training computes the gradient of one scalar loss w.r.t. millions or billions of parameters ($m=1$, $n$ huge), so a single reverse pass gives everything. The price is memory: reverse mode must store (or recompute) forward activations.
</details>

**2. You wrote a custom op with a hand-written backward. How do you test it?**
<details><summary>Answer</summary>

Gradient-check it in float64: compare the analytic gradient with central differences $\frac{f(x+h)-f(x-h)}{2h}$ using a relative error, at random inputs away from kinks (e.g. `torch.autograd.gradcheck`). Also test broadcasting shapes, edge cases (zeros, large values) and that the gradient's shape equals the input's shape.
</details>

**3. Why must you call `optimizer.zero_grad()` in PyTorch, and when would you deliberately not?**
<details><summary>Answer</summary>

`.grad` accumulates across `backward()` calls (gradients are summed, as the chain rule requires within one graph). Without zeroing, step $t$ would use the sum of all previous gradients. You deliberately skip zeroing for **gradient accumulation**: run several micro-batches' `backward()` and step once, simulating a larger batch under a memory limit.
</details>

## 🐝 Fun fact
Reverse-mode automatic differentiation is older than deep learning's backprop papers: Seppo Linnainmaa described it in his 1970 master's thesis at the University of Helsinki — 16 years before Rumelhart, Hinton & Williams popularized backpropagation for neural networks.

## 📚 Further reading
- A. Karpathy, *micrograd* (GitHub, 2020) and the lecture "The spelled-out intro to neural networks and backpropagation: building micrograd" (2022).
- A. G. Baydin, B. A. Pearlmutter, A. A. Radul, J. M. Siskind, "Automatic Differentiation in Machine Learning: a Survey", *JMLR* 18 (2018).
- D. E. Rumelhart, G. E. Hinton, R. J. Williams, "Learning representations by back-propagating errors", *Nature* 323 (1986).
- A. Griewank, A. Walther, *Evaluating Derivatives: Principles and Techniques of Algorithmic Differentiation*, 2nd ed., SIAM (2008).
- A. Paszke et al., "Automatic differentiation in PyTorch", NIPS Autodiff Workshop (2017).
- PyTorch docs: "Autograd mechanics" (pytorch.org/docs/stable/notes/autograd.html).